In [ ]:
import requests
import json
import time
import pandas as pd

since raverly caps results to 100,000, I must creativley splice around this problem
lucky for me a projects either has a photo or doesnt, so thats one easy way to avoid duplicates, things have a rating 0-4 or they dont

first up lets get all US crochet projects with no photo (also spliced by rating)


In [ ]:
url = f"https://api.ravelry.com/projects/search.json"
uid, pw = "read-9add40de0449b103573f7fc8b71c002d", "AmETw80XYOTeWthpmGBbcmSqjCcVHwnq3dvvkVYi"


In [ ]:
#https://www.ravelry.com/projects/search#craft=crochet&photo=no&country=united-states&happiness=0&sort=completed&view=cards
#3173
response = requests.get(url, auth=(uid, pw),params={ "page": 1, "page_size": 3173,"craft":"crochet", "photo":"no","country":"united-states","happiness":"0"} )#,"include": "packs"

In [ ]:
responselist = json.loads(response.text)
df0 = pd.json_normalize(responselist['projects'])

In [ ]:
responselist['paginator']


{'page_count': 1,
 'page': 1,
 'page_size': 3173,
 'results': 3172,
 'last_page': 1}

In [ ]:
#https://www.ravelry.com/projects/search#craft=crochet&photo=no&country=united-states&happiness=1&sort=completed&view=cards
#2553
response = requests.get(url, auth=(uid, pw),params={ "page": 1, "page_size": 2554,"craft":"crochet", "photo":"no","country":"united-states","happiness":"1"} )

In [ ]:
responselist = json.loads(response.text)
df1 = pd.json_normalize(responselist['projects'])
responselist['paginator']

{'page_count': 1,
 'page': 1,
 'page_size': 2554,
 'results': 2553,
 'last_page': 1}

In [ ]:
#https://www.ravelry.com/projects/search#craft=crochet&photo=no&country=united-states&happiness=2&sort=completed&view=cards
#9,625= 4999 + 4626
response = requests.get(url, auth=(uid, pw),params={ "page": 1, "page_size": 4999,"craft":"crochet", "photo":"no","country":"united-states","happiness":"2"} )
responselist = json.loads(response.text)
df2_1 = pd.json_normalize(responselist['projects'])
responselist['paginator']

{'page_count': 2,
 'page': 1,
 'page_size': 4999,
 'results': 9624,
 'last_page': 2}

In [ ]:
response = requests.get(url, auth=(uid, pw),params={ "page": 2, "page_size": 4999,"craft":"crochet", "photo":"no","country":"united-states","happiness":"2"} )
responselist = json.loads(response.text)
df2_2 = pd.json_normalize(responselist['projects'])
responselist['paginator']

{'page_count': 2,
 'page': 2,
 'page_size': 4999,
 'results': 9624,
 'last_page': 2}

In [ ]:
print(len(set(df2_1['id']) & set(df2_2['id'])))
print(len(df2_1))
print(len(df2_2))

0
4999
4625


In [ ]:
#https://www.ravelry.com/projects/search#craft=crochet&photo=no&country=united-states&happiness=3&sort=completed&view=cards
#47,359 /4999 = 10
#for pages 1-10 do this and append/cooncat to df3 wait 10 seconds betwwen each query
df3 = pd.DataFrame()

for page in range(1, 11):
    print(f"Fetching page {page}...")
    response = requests.get(url, auth=(uid, pw), params={
        "page": page,
        "page_size": 4999,
        "craft": "crochet",
        "photo": "no",
        "country": "united-states",
        "happiness": "3"
    })

    responselist = response.json()
    df_page = pd.json_normalize(responselist['projects'])
    df3 = pd.concat([df3, df_page], ignore_index=True)

    if page < 10:
        time.sleep(10)

print(f"Done. Total rows: {len(df3)}")

In [ ]:
#https://www.ravelry.com/projects/search#craft=crochet&photo=no&country=united-states&happiness=4&sort=completed&view=cards
# 72701/4999= 15

df4 = pd.DataFrame()

for page in range(1, 16):
    print(f"Fetching page {page}...")
    response = requests.get(url, auth=(uid, pw), params={
        "page": page,
        "page_size": 4999,
        "craft": "crochet",
        "photo": "no",
        "country": "united-states",
        "happiness": "4"
    })

    responselist = response.json()
    df_page = pd.json_normalize(responselist['projects'])
    df4 = pd.concat([df4, df_page], ignore_index=True)

    if page < 16:
        time.sleep(10)

print(f"Done. Total rows: {len(df4)}")

Fetching page 1...
Fetching page 2...
Fetching page 3...
Fetching page 4...
Fetching page 5...


/tmp/ipykernel_18938/1669700342.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df4 = pd.concat([df4, df_page], ignore_index=True)


Fetching page 6...
Fetching page 7...
Fetching page 8...
Fetching page 9...
Fetching page 10...
Fetching page 11...
Fetching page 12...
Fetching page 13...
Fetching page 14...
Fetching page 15...
Done. Total rows: 72647


In [ ]:
#https://www.ravelry.com/projects/search#craft=crochet&happiness=-0%2B-4%2B-3%2B-2%2B-1&photo=no&finished-in=2025%7C1%7C2026&country=united-states&sort=completed&view=cards
#happiness=-0+-4+-3+-2+-1
#35,682/ 4999=8
df_not_until_2024 = pd.DataFrame()

for page in range(1, 9):
    print(f"Fetching page {page}...")
    response = requests.get(url, auth=(uid, pw), params={
        "page": page,
        "page_size": 4999,
        "craft": "crochet",
        "photo": "no",
        "country": "united-states",
        "happiness": "-0+-4+-3+-2+-1",

    })

    responselist = response.json()
    df_page = pd.json_normalize(responselist['projects'])
    df_not = pd.concat([df_not, df_page], ignore_index=True)

    if page < 8:
        time.sleep(10)

print(f"Done. Total rows: {len(df_not)}")

In [ ]:
#https://www.ravelry.com/projects/search#craft=crochet&happiness=-0%2B-4%2B-3%2B-2%2B-1&photo=no&finished-in=2025%7C1%7C2026&country=united-states&sort=completed&view=cards
#happiness=-0+-4+-3+-2+-1
#35,682/ 4999=8
df_not_2025_2026 = pd.DataFrame()

for page in range(1, 9):
    print(f"Fetching page {page}...")
    response = requests.get(url, auth=(uid, pw), params={
        "page": page,
        "page_size": 4999,
        "craft": "crochet",
        "photo": "no",
        "country": "united-states",
        "happiness": "-0+-4+-3+-2+-1",

    })

    responselist = response.json()
    df_page = pd.json_normalize(responselist['projects'])
    df_not = pd.concat([df_not, df_page], ignore_index=True)

    if page < 8:
        time.sleep(10)

print(f"Done. Total rows: {len(df_not)}")

Fetching page 1...
Fetching page 2...
Fetching page 3...
Fetching page 4...
Fetching page 5...


/tmp/ipykernel_18938/276332353.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_not = pd.concat([df_not, df_page], ignore_index=True)


Fetching page 6...
Fetching page 7...
Fetching page 8...
Done. Total rows: 39992


In [ ]:
df=pd.DataFrame

In [ ]:
df['id'].duplicated().sum()

np.int64(0)

In [ ]:
#df['completed'].value_counts()
df['completed'].value_counts(dropna=False)

,count
completed,
None,67934
2011/01/01,234
2012/12/01,224
2012/01/01,222
2010/01/01,220
...,...
2005/02/19,1
1995/12/15,1
2003/11/22,1


In [ ]:
df.to_csv("US_crochet_no_photo_175323.csv")